## GARCH

Как работает твой GARCH baseline


mean="Constant": 

$r_t = \mu + \epsilon_t$


mean="AR", lags=1: 

$r_t = \mu +  \phi r_{t - 1} + \epsilon_t$

**Volatility equation**

И дальше шум моделируется как:

$\epsilon_t = \sigma_t + z_t, z_t \approx N(0, 1)$


а условная дисперсия:

$\sigma_t^2 = \omega + \alpha \epsilon_{t-1} + \beta \sigma_{t-1}^2$


Это классический GARCH(1,1).

---



### Что именно он предсказывает

Важно: GARCH по своей природе особенно хорош в прогнозе волатильности, а не среднего доходности.

То есть, forecast mean returns у него часто близки к нулю, а вот forecast volatility — содержательная часть модели.

Поэтому если сравнивать DL именно по predicted returns, то неудивительно, что:

* GARCH даёт довольно “плоские” прогнозы,
* DL на дальнем горизонте тоже может тянуться к маленьким значениям.


### Что происходит при multi-step forecast

Если строить прогноз на горизонт H=64, то в момент времени t модель выдаёт:

$\hat{r}_{t + 1 | t}, \hat{r}_{t + 2 | t}, \hat{r}_{t + 3 | t}..., \hat{r}_{t + H | t}$	​


Для AR(1)-части они вычисляются рекурсивно:

$$
\hat{r}_{t + 1 | t} = \mu + \phi r_t \\
\hat{r}_{t + 2 | t} = \mu + \phi \hat{r}_{t + 1 | t}
$$

и так далее.

Если $∣\phi∣<1$, прогнозы mean быстро стремятся к long-run mean, то есть становятся почти плоскими.
---

### Усреднение - Случай. horizon > step

Например:

- horizon = 64
- step = 1

Тогда окна перекрываются.

Пример:

- окно 1 предсказывает дни 1–64,
- окно 2 предсказывает дни 2–65,
- окно 3 предсказывает дни 3–66.

Тогда одна и та же дата может быть предсказана много раз.

**Вариант A:** усреднять

Для каждой `date + ticker` взять среднее всех прогнозов.

Это удобно для:

* построения smooth daily series,
* красивых графиков.

**Вариант B:** брать самый свежий прогноз

То есть для каждой даты брать прогноз из окна с максимальным forecast_origin_date.

Это экономически более осмысленно, если хочется симулировать реальный риск-мониторинг.

---

### Рекомендацию по GARCH для сравнения

**Для честного сравнения exact 64-step window**

- horizon = 64
- step = 64
- не усреднять (last = fresh).

Это идеально для:

* сравнения одного DL окна и одного GARCH окна,
* чистого анализа forecast trajectories.
* Для построения ежедневной риск-серии

**Другой вариант**

- horizon = 64
- step = 1
- усреднять (mean) |  last

**Для самой чистой operational baseline**

- horizon = 1
- step = 1
- last

Это самый понятный режим: каждый день модель предсказывает следующий день.

---

## Другие бейзлайны

### OLS

Обычная линейная регрессия:

features = past returns + optional market snapshot + optional macro snapshot

target = сразу весь horizon

### Ridge

То же, что OLS, но с L2-регуляризацией. Полезно, если много correlated features.

### PCR

Шаги:

- стандартизация признаков
- PCA
- линейная регрессия на главных компонентах

Полезно, если лагов много и есть сильная мультиколлинеарность.

### PLS

Чем-то похожа на PCR, но компоненты строятся уже с учётом target. Для forecasting часто хороший baseline.

### RF (Random Forest Regressor)

- нелинейный baseline
- может ловить interactions и threshold effects

### predict_zero

Всегда предсказывает ноль.

### predict_previous

Берёт последний observed return и повторяет его на весь horizon.

### linear_market

Линейная регрессия только на market_ret_1d snapshot.